In [2]:
# ========== 导入：环境变量工具 + OpenAI SDK ==========

# 标准库 os：后面用 os.getenv 读取 OPENAI_API_KEY
import os
# load_dotenv：从 .env 加载密钥到进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# OpenAI 客户端类：调用 Chat Completions
from openai import OpenAI


In [3]:
# ========== 环境检查：确认 .env 里有没有 API Key ==========

# override=True：已有同名环境变量时仍以 .env 为准
load_dotenv(override=True)
# 取出密钥；后面 OpenAI() 也会读同一环境变量
api_key = os.getenv('OPENAI_API_KEY')

# 没有 key：打印原英文提示（影响用户排查的文案保持原样）
if not api_key:
    print("No API key was found - check your .env file")
else:
    # 有 key：成功提示同样保留英文原文
    print("API key found and looks good so far!")


API key found and looks good so far!


In [4]:
# ========== 创建 OpenAI 客户端实例 ==========

# 无参构造：密钥默认来自环境变量 OPENAI_API_KEY
openai = OpenAI()


In [5]:
# ========== 抓取金融首页正文，并拼一个「通用摘要」user prompt ==========

# 标准库 sys：用来改模块搜索路径，以便 import 课程 week1 的 scraper
import sys
# 把相对路径 ../../week1 加进 sys.path，才能找到 scraper.py
sys.path.append("../../week1")
# 从课程提供的 scraper 导入 fetch_website_contents：HTTP/解析网页正文
from scraper import fetch_website_contents

# 目标站点：Yahoo Finance 首页（URL 字符串保持原样）
url = "https://finance.yahoo.com/"
# 拉取并清洗页面内容，得到可放进 prompt 的文本
website = fetch_website_contents(url)

# 用 f-string 把网页正文嵌进 user prompt；发给模型的英文指令不翻译
user_prompt = f"""
Here are the contents of a website.
{website}
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""


In [6]:
# ========== 金融简报：system +（本格内）user → messages → 调用模型 → 打印 ==========

# 第 1 步：创建提示 —— system 定「尖锐、可执行的金融简报」风格（英文原文保留）
system_prompt = """
You are a sharp, no-nonsense financial assistant that reads market and
financial news summaries and distills them into clear, actionable briefings
to help inform stock decisions. Respond in markdown. Do not wrap the markdown in a code block — respond just
with the markdown.
"""
# 本格内重新定义 user_prompt：带完整 Yahoo Finance 追踪链接；与上格变量同名但内容不同（逻辑保持原样）
user_prompt = """
    Here are the contents of a website, https://finance.yahoo.com/?guccounter=1&guce_referrer=aHR0cHM6Ly93d3cuZ29vZ2xlLmNvbS8&guce_referrer_sig=AQAAAMV993z8g2qQx-YLVzlXZu2zmCHdFgLFRC6LHfZPxDtXkB3kMU4gdGrRyILuTzU_jiAOVB3YJ-yuPdJTz06IEOjSKoNKDTmy-kdX3ZftwXjk4rHXvqCHZCF4Y57yWbtPb5SCpXPeyV6XRtcS_aaPa5wTRq3FYby8D_W6rZP2w78R
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

# 第 2 步：创建消息列表 —— system / user 两条，供 Chat Completions 使用
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
] # fill this in

# 第三步：调用 OpenAI —— 模型 gpt-4.1-mini；非流式一次返回
response = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)
# 取出第一条回复文本（表达式求值结果会出现在笔记本输出里，若该格有展示）
response.choices[0].message.content

# 第四步：打印结果 —— 把同一段内容显式 print 到标准输出
print(response.choices[0].message.content)


# Yahoo Finance Summary

Yahoo Finance is a comprehensive financial news platform offering up-to-date market data, stock quotes, business news, and investment insights.

### Key Features:
- Real-time stock market data and quotes
- Financial news covering major economic events, corporate earnings, market trends, and investment strategies
- Tools for portfolio tracking and stock screeners
- Analysis and opinion pieces from financial experts

### Recent News & Announcements (Typical Content):
- Updates on key market indices (Dow Jones, S&P 500, Nasdaq)
- Earnings reports from major corporations
- Economic indicators releases (inflation data, unemployment rates)
- Government and central bank policy developments
- Industry-specific developments impacting stocks (tech, energy, finance, etc.)

This site is essential for investors seeking timely financial news and actionable market data to inform their stock decisions.
